In [ ]:
import pandas as pd

# path of uploaded file
file_path = "/content/ERA5_daily_2020.csv"

df = pd.read_csv(file_path)

# remove unnecessary columns
df = df.drop(columns=["system:index",".geo"], errors="ignore")

# convert precipitation from meters → mm
df["P_ERA5_daily"] = df["precipitation"] * 1000

# convert temperature from Kelvin → Celsius
df["T2M_ERA5_daily"] = df["temperature"] - 273.15

# keep only required columns
df = df[["date","P_ERA5_daily","T2M_ERA5_daily"]]

# convert date format
df["date"] = pd.to_datetime(df["date"])

# create lag features
df["P_lag1"] = df["P_ERA5_daily"].shift(1)
df["P_lag2"] = df["P_ERA5_daily"].shift(2)
df["P_lag3"] = df["P_ERA5_daily"].shift(3)

# 7-day rolling precipitation
df["P_7day"] = df["P_ERA5_daily"].rolling(7).sum()

# save cleaned dataset
output_file = "/content/ERA5_daily_2020_clean.csv"
df.to_csv(output_file, index=False)

print("Saved:", output_file)
print(df.head())

Saved: /content/ERA5_daily_2020_clean.csv
        date  P_ERA5_daily  T2M_ERA5_daily    P_lag1    P_lag2    P_lag3  \
0 2020-01-01      0.000322        6.688996       NaN       NaN       NaN   
1 2020-01-02      0.000697        9.904776  0.000322       NaN       NaN   
2 2020-01-03      0.007241        7.782444  0.000697  0.000322       NaN   
3 2020-01-04      0.001251        4.909577  0.007241  0.000697  0.000322   
4 2020-01-05      0.000643        7.831264  0.001251  0.007241  0.000697   

   P_7day  
0     NaN  
1     NaN  
2     NaN  
3     NaN  
4     NaN  


In [ ]:
import pandas as pd

# input file
file_path = "/content/Sentinel2_NDVI_NDMI_2020.csv"

df = pd.read_csv(file_path)

# remove unnecessary columns
df = df.drop(columns=["system:index",".geo"], errors="ignore")

# rename columns
df = df.rename(columns={
    "NDVI":"NDVI_S2",
    "NDMI":"NDMI_S2"
})

# convert date
df["date"] = pd.to_datetime(df["date"])

# remove duplicate dates (multiple satellite passes)
df = df.groupby("date").mean().reset_index()

# optional vegetation smoothing (7-day rolling)
df["NDVI_7day"] = df["NDVI_S2"].rolling(7).mean()
df["NDMI_7day"] = df["NDMI_S2"].rolling(7).mean()

# save cleaned dataset
output_file = "/content/Sentinel2_daily_2020_clean.csv"
df.to_csv(output_file, index=False)

print(df.head())
print("Saved:", output_file)

        date   NDMI_S2   NDVI_S2  NDVI_7day  NDMI_7day
0 2020-01-05 -0.170830  0.312395        NaN        NaN
1 2020-01-20 -0.144939  0.318012        NaN        NaN
2 2020-02-14 -0.158074  0.303739        NaN        NaN
3 2020-02-19 -0.111311  0.194361        NaN        NaN
4 2020-03-05 -0.158096  0.336194        NaN        NaN
Saved: /content/Sentinel2_daily_2020_clean.csv


In [ ]:
import pandas as pd
import re

file_path = "/content/USCRN_USCRN_Stillwater-5-WNW_sm_0.050000_0.050000_Stevens-Hydraprobe-II-Sdi-12_20200101_20201231.stm"

# Read file line-by-line, keep only rows that start with a date like 2020/01/01
rows = []
with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if re.match(r"^\d{4}/\d{2}/\d{2}\s+\d{2}:\d{2}", line):
            rows.append(line)

# Now parse the cleaned rows (whitespace separated)
df = pd.read_csv(
    pd.io.common.StringIO("\n".join(rows)),
    sep=r"\s+",
    header=None
)

# Expected columns in your screenshot:
# 0=date 1=time 2=soil_moisture 3=qc_flag 4=measurement_flag
df = df.iloc[:, :5]
df.columns = ["date", "time", "soil_moisture", "qc", "mflag"]

# Keep only good quality measurements
df = df[df["qc"] == "G"].copy()

# Parse datetime with explicit format
df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"], format="%Y/%m/%d %H:%M")

# Daily aggregation (mean)
df["date"] = df["datetime"].dt.date
daily = df.groupby("date", as_index=False)["soil_moisture"].mean()
daily = daily.rename(columns={"soil_moisture": "SM_ISMN_daily"})

# Save
out = "/content/ISMN_daily_2020.csv"
daily.to_csv(out, index=False)

print(daily.head())
print("Saved:", out)
print("Rows:", len(daily))

         date  SM_ISMN_daily
0  2020-01-01       0.288154
1  2020-01-02       0.288261
2  2020-01-03       0.283417
3  2020-01-04       0.274929
4  2020-01-05       0.270591
Saved: /content/ISMN_daily_2020.csv
Rows: 354


In [ ]:
import pandas as pd

# load sentinel file
file_path = "/content/Sentinel2_daily_2020_clean.csv"
df = pd.read_csv(file_path)

# convert date column
df["date"] = pd.to_datetime(df["date"])

# sort values
df = df.sort_values("date")

# set date as index
df = df.set_index("date")

# create full daily date range
daily_index = pd.date_range(start="2020-01-01", end="2020-12-31", freq="D")

# reindex to daily timeline
df_daily = df.reindex(daily_index)

# interpolate missing values (linear interpolation)
df_daily["NDVI_S2"] = df_daily["NDVI_S2"].interpolate(method="linear")
df_daily["NDMI_S2"] = df_daily["NDMI_S2"].interpolate(method="linear")

# reset index
df_daily = df_daily.reset_index()
df_daily = df_daily.rename(columns={"index":"date"})

# save output
output_file = "/content/Sentinel2_daily_2020_interpolated.csv"
df_daily.to_csv(output_file, index=False)

print(df_daily.head())
print("Saved:", output_file)

        date  NDMI_S2   NDVI_S2  NDVI_7day  NDMI_7day
0 2020-01-01      NaN       NaN        NaN        NaN
1 2020-01-02      NaN       NaN        NaN        NaN
2 2020-01-03      NaN       NaN        NaN        NaN
3 2020-01-04      NaN       NaN        NaN        NaN
4 2020-01-05 -0.17083  0.312395        NaN        NaN
Saved: /content/Sentinel2_daily_2020_interpolated.csv


In [ ]:
from google.colab import drive
drive.flush_and_unmount()

Drive not mounted, so nothing to flush and unmount.


In [ ]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
from glob import glob
import subprocess

src = "/content/drive/MyDrive/SMAP_2024"
dst = "/content/SMAP_2024"

os.makedirs(dst, exist_ok=True)

files = glob(src + "/*.h5")
print("Total files:", len(files))

for f in files:
    cmd = f'cp "{f}" "{dst}/"'
    subprocess.run(cmd, shell=True)

print("Copied files:", len(glob(dst+"/*.h5")))

Total files: 366
Copied files: 4


In [ ]:
from glob import glob
len(glob("/content/SMAP_2024/*.h5"))

4

In [ ]:
import h5py
import numpy as np
import pandas as pd
import os
import re
from glob import glob

lat_target = 36.1346
lon_target = -97.1082

folder = "/content/SMAP_2024"

files = sorted(glob(folder + "/*.h5"))

records = []

for file in files:

    name = os.path.basename(file)

    date_str = re.search(r"\d{8}", name).group()
    date = pd.to_datetime(date_str, format="%Y%m%d")

    with h5py.File(file, "r") as f:

        grp = "Soil_Moisture_Retrieval_Data_AM"

        sm  = f[f"{grp}/soil_moisture"][:]
        lat = f[f"{grp}/latitude"][:]
        lon = f[f"{grp}/longitude"][:]
        qc  = f[f"{grp}/retrieval_qual_flag"][:]

        dist = (lat - lat_target)**2 + (lon - lon_target)**2
        idx = np.unravel_index(np.argmin(dist), dist.shape)

        sm_value = sm[idx]
        qc_value = qc[idx]

        if qc_value != 0:
            sm_value = np.nan

    records.append({
        "date": date,
        "SM_SMAP": sm_value
    })

df = pd.DataFrame(records)

df = df.sort_values("date")

df["SM_SMAP_lag1"] = df["SM_SMAP"].shift(1)

df.to_csv("/content/SMAP_daily_2020.csv", index=False)

print(df.head())
print("Rows:", len(df))

OSError: Unable to synchronously open file (truncated file: eof = 7077888, sblock->base_addr = 0, stored_eof = 32339797)

In [ ]:
import pandas as pd

# load
ismn = pd.read_csv("/content/ISMN_daily_2021-2024.csv")
smap = pd.read_csv("/content/SMAP_daily_2021-2024.csv")
era5 = pd.read_csv("/content/ERA5_daily_2021_2024_clean.csv")
s2   = pd.read_csv("/content/Sentinel2_daily_2021_interpolated.csv")

# parse dates
for df in [ismn, smap, era5, s2]:
    df["date"] = pd.to_datetime(df["date"])

# merge on date (keep full daily index from ISMN or full year)
merged = ismn.merge(smap, on="date", how="left") \
             .merge(s2,   on="date", how="left") \
             .merge(era5, on="date", how="left")

# optional: add SMAP lag already exists; ensure types numeric
num_cols = [c for c in merged.columns if c != "date"]
merged[num_cols] = merged[num_cols].apply(pd.to_numeric, errors="coerce")

# save
out = "/content/fusion_training_table_2020.csv"
merged.to_csv(out, index=False)

print("Saved:", out)
print(merged.head(10))
print("Rows:", len(merged), "Cols:", len(merged.columns))

Saved: /content/fusion_training_table_2020.csv
        date  SM_ISMN_daily   SM_SMAP  SM_SMAP_lag1   NDMI_S2   NDVI_S2  \
0 2021-01-01       0.360000  0.092000           NaN       NaN       NaN   
1 2021-01-02       0.435158       NaN      0.092000       NaN       NaN   
2 2021-01-03       0.395778       NaN           NaN       NaN       NaN   
3 2021-01-04       0.351091       NaN           NaN -0.103337  0.345999   
4 2021-01-05       0.322538  0.319678           NaN -0.107518  0.342547   
5 2021-01-06       0.307542  0.198334      0.319678 -0.111700  0.339096   
6 2021-01-07       0.297000       NaN      0.198334 -0.115881  0.335645   
7 2021-01-08       0.289792  0.296406           NaN -0.120062  0.332194   
8 2021-01-09       0.272714       NaN      0.296406 -0.124243  0.328743   
9 2021-01-10       0.266375  0.292853           NaN -0.128424  0.325291   

   NDVI_7day  NDMI_7day  P_ERA5_daily  T2M_ERA5_daily     P_lag1     P_lag2  \
0        NaN        NaN     29.166380        0.5

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# load merged dataset
df = pd.read_csv("/content/fusion_training_table_2020.csv")

df["date"] = pd.to_datetime(df["date"])

# sort
df = df.sort_values("date")

# fill missing values
df = df.fillna(method="ffill")
df = df.fillna(method="bfill")

# numeric columns only
num_cols = df.columns.drop("date")

# normalize 0–1
scaler = MinMaxScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# save
df.to_csv("/content/fusion_training_table_2020_normalized.csv", index=False)

print(df.head())
print("Rows:", len(df))

        date  SM_ISMN_daily   SM_SMAP  SM_SMAP_lag1   NDMI_S2   NDVI_S2  \
0 2021-01-01       0.771229  0.157137      0.157137  0.161459  0.500055   
1 2021-01-02       0.968106  0.157137      0.157137  0.161459  0.500055   
2 2021-01-03       0.864949  0.157137      0.157137  0.161459  0.500055   
3 2021-01-04       0.747891  0.157137      0.157137  0.161459  0.500055   
4 2021-01-05       0.673098  0.791275      0.157137  0.151757  0.495184   

   NDVI_7day  NDMI_7day  P_ERA5_daily  T2M_ERA5_daily    P_lag1    P_lag2  \
0        0.0   0.613401      0.594302        0.149275  0.594302  0.594302   
1        0.0   0.613401      0.038508        0.124152  0.594302  0.594302   
2        0.0   0.613401      0.000389        0.139345  0.038508  0.594302   
3        0.0   0.613401      0.000008        0.220433  0.000389  0.038508   
4        0.0   0.613401      0.000009        0.268991  0.000008  0.000389   

     P_lag3    P_7day  
0  0.594302  0.384147  
1  0.594302  0.384147  
2  0.594302  0

/tmp/ipykernel_1827/3163161464.py:13: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill")
/tmp/ipykernel_1827/3163161464.py:14: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="bfill")


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving fusion_training_table_2021_2024_normalized.csv to fusion_training_table_2021_2024_normalized (1).csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving fusion_training_table_2020_normalized.csv to fusion_training_table_2020_normalized (1).csv


In [ ]:
import pandas as pd

# Read uploaded files (use exact filenames shown after upload)
df1 = pd.read_csv('fusion_training_table_2020_normalized.csv')
df2 = pd.read_csv('fusion_training_table_2021_2024_normalized.csv')

# Combine row-wise
combined_df = pd.concat([df1, df2], ignore_index=True)

# Preview
combined_df.head()

,date,SM_ISMN_daily,SM_SMAP,SM_SMAP_lag1,NDMI_S2,NDVI_S2,NDVI_7day,NDMI_7day,P_ERA5_daily,T2M_ERA5_daily,P_lag1,P_lag2,P_lag3,P_7day
0,2020-01-01,0.527622,0.653687,0.653687,0.0,0.396278,0.0,0.014187,0.000007,0.241676,0.000007,0.000007,0.000007,0.000099
1,2020-01-02,0.527941,0.653687,0.653687,0.0,0.396278,0.0,0.014187,0.000016,0.340968,0.000007,0.000007,0.000007,0.000099
2,2020-01-03,0.513520,0.619134,0.653687,0.0,0.396278,0.0,0.014187,0.000168,0.275438,0.000016,0.000007,0.000007,0.000099
3,2020-01-04,0.488252,0.337482,0.619134,0.0,0.396278,0.0,0.014187,0.000029,0.186733,0.000168,0.000016,0.000007,0.000099
4,2020-01-05,0.475339,0.337482,0.337482,0.0,0.396278,0.0,0.014187,0.000015,0.276945,0.000029,0.000168,0.000016,0.000099


In [ ]:
# Save file
combined_df.to_csv('final_combined_dataset.csv', index=False)

# Download to your system
files.download('final_combined_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>